## Data Preprocessing

In [2]:
import pandas as pd
import numpy as np
import csv
import os
import nltk
from nltk.corpus import wordnet as wn
import random
from transformers import BertTokenizer, BertModel
import torch 

nltk.download('wordnet')


C:\Users\rldor\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

### Cleaning Data

In this section, we reformat the initial files by getting rid off unneeded columns in certain rows which may lead to errors. In addition, the words are all put into one file "words.csv" making them easier to deal with in the next part.

In [ ]:
nltk.download('wordnet')


In [44]:
files = ['Data/adjectives.csv', 'Data/adverbs.csv', 'Data/nouns.csv', 'Data/verbs.csv']

# This goes through a file and keeps just the first column of each one
def clean_csv(input_file):
    temp_file = input_file + ".csv"
    with open(input_file, "r", newline="", encoding="utf-8") as infile, \
        open(temp_file, "w", newline="", encoding="utf-8") as outfile:

        reader = csv.reader(infile)
        writer = csv.writer(outfile)

        for row in reader:
            if row:
                writer.writerow([row[0]])

    os.replace(temp_file, input_file)

# This takes the files and turns them into a single csv file with two columns, one for the word and one for the part of speech
def files_to_csv(files):
    df = pd.DataFrame()

    for file in files:
        clean_csv(file)
        pos = os.path.basename(file).split('.')[0]
        temp_df = pd.read_csv(file, header=None, names=['word'])
        temp_df['pos'] = pos
        df = pd.concat([df, temp_df], ignore_index=True)

    df.to_csv('Data/words.csv', index=False)

# Executing the functions on "files" list
files_to_csv(files)

### Removing Unneeded and Weird Words

This is being done to preseve a balance between the numbers of words in each class as well as removing strange words (some of which, especially for nouns, are just strings of numbers).

In [ ]:
# This function reads the csv and deletes rows where the word contains either a space a or a digit, since we only want single words
def clean_words(input_file):
    df = pd.read_csv(input_file)
    df = df.dropna(subset=['word'])
    df = df[~df['word'].str.contains(' ')]
    df = df[~df['word'].str.contains(r'\d', regex=True)]
    df.to_csv(input_file, index=False)

clean_words('Data/words.csv')



### Stabalizing Classes

Because there is a lot of imbalance in the number of words in each class, I needed to get rid of some and add other. To do this I added verbs from the NLTK library "wordnet" because my original dataset had only 125 or so. I also, for the other parts of speech, randomly sampled from these classes so as to only take as many as I needed.

In [ ]:
# adding verbs to words.csv
verbs = set()
for synset in wn.all_synsets(pos=wn.VERB):
    for lemma in synset.lemmas():
        verbs.add(lemma.name())

verbs = [v for v in verbs if '_' not in v]

print(f"Number of verbs from WordNet: {len(verbs)}")

words = pd.read_csv('Data/words.csv')
print("csv length before adding verbs:", len(words))

for word in verbs:
    if word not in words['word'].values:
        words.loc[len(words)] = {'word': word, 'pos': 'verbs'}
words.to_csv('Data/words.csv', index=False)

# This reads the csv and evens out the number of words in each pos by randomly sampling from each
words = pd.read_csv('Data/words.csv')

min_count = words['pos'].value_counts().min()

# Sampling from each one
final_df = (
    words
    .groupby('pos', group_keys=False)
    .apply(lambda x: x.sample(n=min_count, random_state=42))
)

final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

final_df.to_csv('Data/words.csv', index=False)


Number of verbs from WordNet: 8702
csv length before adding verbs: 176827
csv length after adding verbs: 181206
pos
nouns         141897
adjectives     28479
adverbs         6276
verbs           4554
Name: count, dtype: int64
final_df length before shuffling: 18216
18216 words in final csv


C:\Users\rldor\AppData\Local\Temp\ipykernel_17264\2438404659.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_count, random_state=42))


### Words to Embeddings

In this section we take each of the words and map them to embeddings which we will use later on for our machine learning process and catagorizing words based on part of speech. I am using the BERT model by Google from HuggingFace to do tokenization and create embeddings for each of the words.

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

df_words = pd.read_csv('Data/words.csv')
data = []

for index, row in df_words.iterrows():
    word = row['word']
    pos = row['pos']
    
    inputs = tokenizer(word, return_tensors='pt')

    with torch.no_grad():
        embeddings = model.embeddings(
            input_ids=inputs['input_ids'],
            token_type_ids=inputs['token_type_ids']
        )

    embeddings = embeddings[0][:-1]  # keep CLS but drop SEP tokens
    data.append({'word': word, 'pos': pos, 'embedding': embeddings})

torch.save(data, 'Data/word_embeddings.pt')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 918.27it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [1]:
import pandas as pd

df = pd.read_csv('Data/words.csv')
print(df.head())

              word         pos
0          halacha       nouns
1          Flobert       nouns
2          caulker       nouns
3      ingeniously     adverbs
4  interscholastic  adjectives
